In [11]:
ruta_embeddings_gen = "d-mercator/graphs/"


path_emb_9n = ruta_embeddings_gen + "9n/437037/2/"
path_emb_ch = ruta_embeddings_gen + "ch/394718/2/"
path_emb_nat = ruta_embeddings_gen + "nat/429624/2/"

In [12]:
import math
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd

from GRH_all import dhyp, greedy_route, shortest_path_length

Position = Tuple[float, float]
Adjacency = List[List[int]]


In [13]:
def load_dmercator_embedding(folder: str, graph_id: str) -> Tuple[Adjacency, Dict[int, Position], int]:
    """Load a D-Mercator S1/H2 embedding and remap labels to contiguous 1..N indices."""
    folder_path = Path(folder)
    coord_path = folder_path / f"{graph_id}.inf_coord"
    edge_path = folder_path / f"{graph_id}.edge"

    labels = []
    positions_raw: Dict[str, Position] = {}

    with open(coord_path, "r", encoding="utf-8") as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            parts = stripped.split()
            if len(parts) < 4:
                continue
            label = parts[0]
            theta_rad = float(parts[2])
            hyp_rad = float(parts[3])
            labels.append(label)
            positions_raw[label] = (hyp_rad, math.degrees(theta_rad))

    edges: List[Tuple[str, str]] = []
    with open(edge_path, "r", encoding="utf-8") as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            parts = stripped.split()
            if len(parts) < 2:
                continue
            i_label, j_label = parts[0], parts[1]
            edges.append((i_label, j_label))
            if i_label not in positions_raw:
                labels.append(i_label)
            if j_label not in positions_raw:
                labels.append(j_label)

    unique_labels = list(dict.fromkeys(labels))
    label_to_idx = {label: idx + 1 for idx, label in enumerate(unique_labels)}
    n_nodes = len(unique_labels)

    adjacency: Adjacency = [[] for _ in range(n_nodes + 1)]
    for i_label, j_label in edges:
        i = label_to_idx[i_label]
        j = label_to_idx[j_label]
        adjacency[i].append(j)
        adjacency[j].append(i)

    positions = {
        label_to_idx[label]: positions_raw.get(label, (0.0, 0.0))
        for label in unique_labels
    }

    return adjacency, positions, n_nodes


In [14]:
def compute_navigability(
    adjacency: Adjacency,
    positions: Dict[int, Position],
    diam: int = 1000,
    pair_step: int = 1,
) -> dict:
    """Compute hyperbolic greedy routing statistics over sampled node pairs."""
    n_nodes = len(adjacency) - 1

    failures = 0
    total_pairs = 0
    identical_positions = 0

    stretch_sum = 0.0
    distance_sum = 0.0
    stretch_sq_sum = 0.0
    distance_sq_sum = 0.0

    for source in range(1, n_nodes + 1):
        for target in range(1, n_nodes + 1, pair_step):
            if source == target:
                continue

            success, greedy_hops, greedy_distance_sum = greedy_route(
                source, target, adjacency, positions, diam=diam
            )

            if not success:
                failures += 1
            else:
                shortest = shortest_path_length(source, target, adjacency, diam=diam)
                if shortest is None or shortest == 0:
                    failures += 1
                else:
                    topo_stretch = greedy_hops / float(shortest)
                    stretch_sum += topo_stretch
                    stretch_sq_sum += topo_stretch ** 2

                    source_pos = positions.get(source, (0.0, 0.0))
                    target_pos = positions.get(target, (0.0, 0.0))
                    if source_pos[0] == target_pos[0] and source_pos[1] == target_pos[1]:
                        identical_positions += 1
                    else:
                        direct_distance = dhyp(
                            source_pos[0], source_pos[1],
                            target_pos[0], target_pos[1],
                        )
                        dist_stretch = greedy_distance_sum / float(direct_distance)
                        distance_sum += dist_stretch
                        distance_sq_sum += dist_stretch ** 2

            total_pairs += 1

    successes = total_pairs - failures
    success_ratio = successes / float(total_pairs) if total_pairs > 0 else float("nan")

    if successes > 0:
        avg_topo_stretch = stretch_sum / float(successes)
        avg_stretch = distance_sum / float(successes)
        std_topo_stretch = math.sqrt(
            max(stretch_sq_sum / float(successes) - avg_topo_stretch ** 2, 0.0)
        )
        std_stretch = math.sqrt(
            max(distance_sq_sum / float(successes) - avg_stretch ** 2, 0.0)
        )
    else:
        avg_topo_stretch = float("nan")
        avg_stretch = float("nan")
        std_topo_stretch = float("nan")
        std_stretch = float("nan")

    return {
        "n_nodes": n_nodes,
        "n_pairs": total_pairs,
        "n_successes": successes,
        "n_failures": failures,
        "success_ratio": success_ratio,
        "avg_stretch": avg_stretch,
        "std_stretch": std_stretch,
        "avg_topo_stretch": avg_topo_stretch,
        "std_topo_stretch": std_topo_stretch,
        "identical_positions": identical_positions,
    }


In [15]:
networks = {
    "9n": (path_emb_9n, "437037"),
    "ch": (path_emb_ch, "394718"),
    "nat": (path_emb_nat, "429624"),
}

results = []
for name, (folder, graph_id) in networks.items():
    print(f"Calculando navegabilidad para {name} ({graph_id})...")
    adjacency, positions, n_nodes = load_dmercator_embedding(folder, graph_id)
    stats = compute_navigability(adjacency, positions, pair_step=1)
    stats["network"] = name
    stats["graph_id"] = graph_id
    stats["folder"] = folder
    results.append(stats)
    print(
        f"  nodos={stats['n_nodes']}, pares={stats['n_pairs']}, "
        f"success={stats['success_ratio']:.4f}, stretch={stats['avg_stretch']:.4f}"
    )

"""columns = [
    "network",
    "graph_id",
    "folder",
    "n_nodes",
    "n_pairs",
    "n_successes",
    "n_failures",
    "success_ratio",
    "avg_stretch",
    "std_stretch",
    "avg_topo_stretch",
    "std_topo_stretch",
    "identical_positions",
]"""


columns = [
    "network",
    "success_ratio",
    "avg_stretch",
    "std_stretch",
    "avg_topo_stretch",
    "std_topo_stretch",
]

df_navigability = pd.DataFrame(results)[columns]
display(df_navigability)

output_path = Path("measures/navigability_embeddings.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
df_navigability.to_csv(output_path, index=False)
print(f"Resultados guardados en {output_path}")


Calculando navegabilidad para 9n (437037)...
  nodos=246, pares=60270, success=0.9500, stretch=1.5433
Calculando navegabilidad para ch (394718)...
  nodos=313, pares=97656, success=0.9551, stretch=1.3463
Calculando navegabilidad para nat (429624)...
  nodos=178, pares=31506, success=0.9832, stretch=1.3871


,network,success_ratio,avg_stretch,std_stretch,avg_topo_stretch,std_topo_stretch
0,9n,0.949992,1.543321,0.364871,1.022527,0.103639
1,ch,0.955057,1.346308,0.245678,1.017459,0.089554
2,nat,0.983241,1.387051,0.240461,1.023396,0.108963


Resultados guardados en measures/navigability_embeddings.csv
